In [0]:
# ============================================
# CELDA 1 — Imports
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from imblearn.combine import SMOTETomek
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas")

✅ Librerías cargadas


In [0]:
# CELDA 2 — Cargar datos
# ============================================
df = pd.read_csv(
    '/Volumes/workspace/default/fraud_data/fraud_transactions.csv'
)
df['label_text'] = df['is_fraud'].map(
    {0:'LEGITIMATE', 1:'FRAUD'}
)

print(f"✅ Datos cargados: {df.shape}")
print(f"   Fraude:    {df['is_fraud'].sum():,}")
print(f"   Legítimas: {(df['is_fraud']==0).sum():,}")

✅ Datos cargados: (100000, 16)
   Fraude:    15,000
   Legítimas: 85,000


In [0]:
# CELDA 3 — Definir features
# ============================================
# Todas las features disponibles
ALL_FEATURES = [
    'transaction_amount',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'is_online',
    'merchant_category',
    'customer_age',
    'account_age_days',
    'transactions_last_24h',
    'avg_transaction_amt',
    'amount_vs_avg_ratio',
    'distance_from_home',
    'failed_attempts',
    'is_foreign_transaction'
]

X_all = df[ALL_FEATURES].values
y_all = df['is_fraud'].values

print(f"✅ Features definidas: {len(ALL_FEATURES)}")
print(f"   {ALL_FEATURES}")

✅ Features definidas: 14
   ['transaction_amount', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_online', 'merchant_category', 'customer_age', 'account_age_days', 'transactions_last_24h', 'avg_transaction_amt', 'amount_vs_avg_ratio', 'distance_from_home', 'failed_attempts', 'is_foreign_transaction']


In [0]:
# CELDA 4 — Selección con Mutual Information
# ============================================
# Mutual Information mide cuánta información aporta
# cada feature sobre el target
# Es mejor que correlación para relaciones no lineales

print("⏳ Calculando Mutual Information...")

# Encode categorical features for mutual information
from sklearn.preprocessing import LabelEncoder
X_encoded = X_all.copy()
for i, feature in enumerate(ALL_FEATURES):
    if df[feature].dtype == 'object':
        le = LabelEncoder()
        X_encoded[:, i] = le.fit_transform(df[feature])

mi_scores = mutual_info_classif(
    X_encoded, y_all, random_state=42
)

mi_df = pd.DataFrame({
    'feature': ALL_FEATURES,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print("\n🧠 Mutual Information Scores:")
print(mi_df.to_string(index=False))

# Gráfica
fig = px.bar(
    mi_df,
    x='mi_score',
    y='feature',
    orientation='h',
    title='🧠 Feature Importance — Mutual Information Score',
    color='mi_score',
    color_continuous_scale='Viridis',
    labels={
        'mi_score':'MI Score',
        'feature':'Feature'
    }
)
fig.update_layout(
    height=600,
    yaxis={'categoryorder':'total ascending'}
)
fig.show()

⏳ Calculando Mutual Information...

🧠 Mutual Information Scores:
               feature  mi_score
 transactions_last_24h  0.387705
    distance_from_home  0.352132
   amount_vs_avg_ratio  0.334789
    transaction_amount  0.260110
       failed_attempts  0.200887
is_foreign_transaction  0.160813
      account_age_days  0.095800
           hour_of_day  0.084949
   avg_transaction_amt  0.074523
             is_online  0.061845
     merchant_category  0.053306
           day_of_week  0.004897
            is_weekend  0.002180
          customer_age  0.000000


In [0]:
# CELDA 5 — Seleccionar top features
# ============================================
# Tomamos todas las features con MI > 0
# En este dataset todas deberían tener valor

TOP_FEATURES = list(
    mi_df[mi_df['mi_score'] > 0]['feature'].values
)

print(f"✅ Features seleccionadas: {len(TOP_FEATURES)}")
print(f"   {TOP_FEATURES}")

X = df[TOP_FEATURES].values
y = df['is_fraud'].values

✅ Features seleccionadas: 13
   ['transactions_last_24h', 'distance_from_home', 'amount_vs_avg_ratio', 'transaction_amount', 'failed_attempts', 'is_foreign_transaction', 'account_age_days', 'hour_of_day', 'avg_transaction_amt', 'is_online', 'merchant_category', 'day_of_week', 'is_weekend']


In [0]:
# CELDA 6 — Train / Validation / Test Split
# ============================================
# stratify=y es CRÍTICO
# Mantiene el 15% de fraude en cada split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print(f"{'Split':<10} {'Total':>8} {'Fraud':>8} {'Fraud%':>8}")
print("-" * 38)
print(f"{'Train':<10} {len(X_train):>8,} {y_train.sum():>8} {y_train.mean()*100:>7.2f}%")
print(f"{'Val':<10} {len(X_val):>8,} {y_val.sum():>8} {y_val.mean()*100:>7.2f}%")
print(f"{'Test':<10} {len(X_test):>8,} {y_test.sum():>8} {y_test.mean()*100:>7.2f}%")

# ============================================

Split         Total    Fraud   Fraud%
--------------------------------------
Train        60,000     9000   15.00%
Val          20,000     3000   15.00%
Test         20,000     3000   15.00%


In [0]:
# CELDA 7 — RobustScaler
# ============================================
# Por qué RobustScaler y no StandardScaler:
# Las transacciones de fraude son outliers por definición
# RobustScaler usa mediana e IQR en lugar de media y std
# Mucho más resistente a valores extremos

# Encode categorical features before scaling
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Find categorical column index
cat_col_idx = None
for i, feature in enumerate(TOP_FEATURES):
    if df[feature].dtype == 'object':
        cat_col_idx = i
        break

# Encode if categorical column exists
if cat_col_idx is not None:
    X_train[:, cat_col_idx] = le.fit_transform(X_train[:, cat_col_idx])
    X_val[:, cat_col_idx] = le.transform(X_val[:, cat_col_idx])
    X_test[:, cat_col_idx] = le.transform(X_test[:, cat_col_idx])

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)  # fit SOLO en train
X_val_scaled   = scaler.transform(X_val)         # solo transform
X_test_scaled  = scaler.transform(X_test)        # solo transform

print("✅ RobustScaler aplicado")
print(f"   Train mean: {X_train_scaled.mean():.4f} (≈0 esperado)")
print(f"   Train std:  {X_train_scaled.std():.4f} (≈1 esperado)")

# ============================================

✅ RobustScaler aplicado
   Train mean: 0.3580 (≈0 esperado)
   Train std:  3.5705 (≈1 esperado)


In [0]:
# CELDA 8 — Visualizar desbalance antes de SMOTE
# ============================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'BEFORE SMOTE-Tomek',
        'AFTER SMOTE-Tomek (preview)'
    )
)

counts_before = pd.Series(y_train).value_counts()
fig.add_trace(
    go.Bar(
        x=['Legitimate','Fraud'],
        y=[counts_before[0], counts_before[1]],
        marker_color=['#27ae60','#e74c3c'],
        name='Before'
    ),
    row=1, col=1
)

print(f"\n📊 ANTES de SMOTE-Tomek:")
print(f"   Legitimate: {counts_before[0]:,}")
print(f"   Fraud:      {counts_before[1]:,}")
print(f"   Ratio:      {counts_before[0]//counts_before[1]}:1")



📊 ANTES de SMOTE-Tomek:
   Legitimate: 51,000
   Fraud:      9,000
   Ratio:      5:1


In [0]:
# CELDA 9 — Aplicar SMOTE-Tomek
# ============================================
# SMOTE: crea ejemplos sintéticos de la clase minoritaria
# Tomek: elimina pares ambiguos de la clase mayoritaria
# Resultado: frontera de decisión más limpia

print("⏳ Aplicando SMOTE-Tomek... (2-3 minutos)")

smote = SMOTETomek(random_state=42)
X_train_res, y_train_res = smote.fit_resample(
    X_train_scaled, y_train
)

counts_after = pd.Series(y_train_res).value_counts()

print(f"\n📊 DESPUÉS de SMOTE-Tomek:")
print(f"   Legitimate: {counts_after[0]:,}")
print(f"   Fraud:      {counts_after[1]:,}")
print(f"   Ratio:      {counts_after[0]//counts_after[1]}:1")

# Completar la gráfica comparativa
fig.add_trace(
    go.Bar(
        x=['Legitimate','Fraud'],
        y=[counts_after[0], counts_after[1]],
        marker_color=['#27ae60','#e74c3c'],
        name='After'
    ),
    row=1, col=2
)
fig.update_layout(
    title='Effect of SMOTE-Tomek on Class Balance',
    height=450,
    showlegend=False
)
fig.show()

⏳ Aplicando SMOTE-Tomek... (2-3 minutos)

📊 DESPUÉS de SMOTE-Tomek:
   Legitimate: 51,000
   Fraud:      51,000
   Ratio:      1:1


In [0]:
# CELDA 10 — Guardar datasets procesados
# ============================================
datasets = {
    'X_train'     : X_train_res,
    'y_train'     : y_train_res,
    'X_val'       : X_val_scaled,
    'y_val'       : y_val,
    'X_test'      : X_test_scaled,
    'y_test'      : y_test,
    'feature_names': TOP_FEATURES,
    'scaler'      : scaler
}

# Save directly to Unity Catalog Volume (serverless-compatible)
with open('/Volumes/workspace/default/fraud_data/datasets.pkl', 'wb') as f:
    pickle.dump(datasets, f)

print("✅ Datasets guardados en Volume")
print(f"\n📋 RESUMEN FINAL:")
print(f"   Train: {X_train_res.shape} | Fraud: {y_train_res.sum():,}")
print(f"   Val:   {X_val_scaled.shape}  | Fraud: {y_val.sum():,}")
print(f"   Test:  {X_test_scaled.shape}  | Fraud: {y_test.sum():,}")
print(f"   Features: {len(TOP_FEATURES)}")
print(f"\n✅ Día 4 completo — Datos listos para modelado")

✅ Datasets guardados en Volume

📋 RESUMEN FINAL:
   Train: (102000, 13) | Fraud: 51,000
   Val:   (20000, 13)  | Fraud: 3,000
   Test:  (20000, 13)  | Fraud: 3,000
   Features: 13

✅ Día 4 completo — Datos listos para modelado
